# TB-Trust — 06: Blind inversion of the phone-capture channel

Given one 8-bit JPEG of a film on a lightbox — no reference shot, no knowledge of the phone,
no calibration target added to the scene — recover:

1. the **veiling glare** field, from the direct-exposure region used as an optical beam stop;
2. the **point-spread function**, from the collimation border as an ISO 12233 slanted edge;
3. the **tone curve**, from two-point densitometry against `D_min` and `D_max`;

and hence the **calibrated optical density** map, which is the radiometrically meaningful
quantity — density differences are proportional to path-integrated X-ray attenuation, and
that is what a radiologist is actually reading. Pixel values are that signal after four
transformations, three unknown and one lossy.

This notebook **scores those recoveries against ground truth**, which is possible because
the forward model in `physics/film.py` knows what it applied. Notebook 07 turns the recovered
channel into a bound; this one establishes that the channel was recovered at all.

In [ ]:
# --- configuration ---------------------------------------------------------
# Every path comes from the environment first, so this notebook runs unmodified
# on Kaggle, locally, or under scripts/test_notebooks.py in CI.
import os

REPO = os.environ.get("TBTRUST_REPO", "/kaggle/working/tb-trust")
DATA = os.environ.get("TBTRUST_DATA", "/kaggle/input/tuberculosis-tb-chest-xray-dataset")
WORK = os.environ.get("TBTRUST_WORK", "/kaggle/working")
REPO_URL = os.environ.get("TBTRUST_REPO_URL", "https://github.com/AIscend-Research/tb-trust.git")

MANIFEST = f"{WORK}/manifest.csv"
OUT = f"{WORK}/outputs"
os.makedirs(OUT, exist_ok=True)

# Working resolution for the physics. This is the single most consequential knob
# in the whole track: the density floor depends on how many pixels a finding
# spans, so a 2 mm miliary nodule is under two pixels at 320 px and the
# certificate correctly -- but uselessly -- calls every image insufficient.
# A phone photographing a 35 cm film at 3000 px gets about 8 px/mm. 1024 is the
# smallest size at which the severity sweep separates properly; drop it only to
# make a CI run cheap.
PHYSICS_SIZE = int(os.environ.get("TBTRUST_PHYSICS_SIZE", "1024"))
N_IMAGES = int(os.environ.get("TBTRUST_PHYSICS_N", "24"))

print("REPO:", REPO, "\nDATA:", DATA, "\nWORK:", WORK)
print("physics size:", PHYSICS_SIZE, " images:", N_IMAGES)

In [ ]:
# Enter the repo and make it importable. The install is skipped when the package
# already resolves, so re-running is cheap.
import importlib.util
import os
import subprocess
import sys

os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
if importlib.util.find_spec("tbtrust") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 110
print("tbtrust ready from", REPO)

## 1. The forward model, one image at a time

Photograph a synthetic film and look at what each stage of the capture did. The film carries the three fiducials; the capture applies geometry, a two-component lens PSF (a narrow core plus
the broad halo that *is* veiling glare), sensor noise, an unknown ISP tone curve, 8-bit
quantisation and JPEG.

In [ ]:
from tbtrust.physics.density import density_to_display
from tbtrust.physics.film import capture, sample_params, synthetic_chest_density

base, ftruth = synthetic_chest_density(size=PHYSICS_SIZE // 2, rng=np.random.default_rng(0))
params = sample_params(0.5, np.random.default_rng(1))
photo, truth = capture(base, params, fiducial_truth=ftruth, rng=np.random.default_rng(2))

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(density_to_display(base), cmap="gray")
axes[0].set_title("film (optical density)")
axes[1].imshow(truth.luminance_true, cmap="gray")
axes[1].set_title("on the lightbox")
axes[2].imshow(truth.glare_field_true, cmap="magma")
axes[2].set_title("true veil (glare)")
axes[3].imshow(photo, cmap="gray")
axes[3].set_title("the phone photo\n(all the estimator sees)")
for a in axes:
    a.axis("off")
fig.tight_layout()
plt.show()

print(f"true PSF sigma      {truth.psf_sigma_effective:.2f} px")
print(f"true veil fraction  {truth.veil_fraction_true:.3f}")
print(f"true tone gamma     {params.tone_gamma:.2f}  (estimator assumes {2.2} with a prior)")

Severity is the axis every result in notebooks 07 and 08 is plotted against, so it is worth
seeing what it means before trusting a curve drawn over it.

In [ ]:
from tbtrust.physics import figures as FIG
from tbtrust.physics.density import density_to_display

FIG.show(FIG.degradation_ladder((density_to_display(base) * 255).astype("uint8"),
                                size=min(512, PHYSICS_SIZE)))
plt.show()

## 2. Invert it blind

The estimators are mutually entangled — the tone curve needs the veil, the veil needs the PSF,
the PSF needs a linearised image — so they run as a short fixed-point iteration rather than a
pipeline. See the module docstring of `physics/invert.py` for why two passes suffice.

In [ ]:
from tbtrust.physics.invert import invert

cal = invert(photo)
pd.Series(cal.summary()).to_frame("value")

In [ ]:
from tbtrust.physics import figures as FIG

# The inversion, end to end: the photograph, the veil measured from the beam stop,
# the contrast compression that veil imposes, and the recovered optical density.
# On a simulated capture the true veil is shown alongside the measured one.
FIG.show(FIG.inversion_panels(photo, cal, truth))
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(cal.veil, cmap="magma")
axes[0].set_title("recovered veil")
axes[1].imshow(truth.glare_field_true, cmap="magma")
axes[1].set_title("true veil")
axes[2].imshow(density_to_display(cal.density), cmap="gray")
axes[2].set_title("recovered density")
im = axes[3].imshow(cal.sigma_random, cmap="viridis")
axes[3].set_title("per-pixel density noise")
plt.colorbar(im, ax=axes[3], fraction=0.046)
for a in axes:
    a.axis("off")
fig.tight_layout()
plt.show()

fr = cal.psf.freqs, cal.psf.mtf
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
if len(fr[0]):
    axes[0].plot(fr[0], fr[1], label="measured (slanted edge)")
g = np.linspace(0, 0.5, 100)
axes[0].plot(g, np.exp(-2 * np.pi**2 * truth.psf_sigma_effective**2 * g**2), "--", label="true Gaussian-equivalent")
axes[0].axhline(0.5, color="k", lw=0.5)
axes[0].set_xlim(0, 0.5)
axes[0].set_ylim(0, 1.05)
axes[0].set_xlabel("cycles / pixel")
axes[0].set_ylabel("MTF")
axes[0].legend(fontsize=8)
axes[0].set_title("Capture MTF from the collimation border")

v = np.linspace(0.01, 1, 200)
axes[1].plot(v, cal.tone.to_luminance(v), label="recovered inverse tone curve")
axes[1].set_xlabel("pixel value")
axes[1].set_ylabel("relative luminance")
axes[1].set_title(f"Tone inversion ({cal.tone.method}, gamma={cal.tone.gamma:.2f})")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

## 3. Recovery across the severity sweep

One image proves nothing. `validate.recovery_experiment` runs the whole loop over many images
and severities and scores every recovered quantity against truth.

The two columns that carry the certificate are `veil_fraction_err` and `psf_sigma_rel_err`.
`density_diff_rmse` is the *differential* density error — the quantity the resolution floor
actually bounds — as opposed to `density_abs_rmse`, which is dominated by the gamma prior and
is reported mainly to show that it is. A common scale error moves a lesion and the lung field
two millimetres away together, and cancels in their difference.

In [ ]:
from tbtrust.physics import validate as V

rec = pd.DataFrame(V.recovery_experiment(
    n_images=max(2, N_IMAGES // 6), severities=(0.0, 0.25, 0.5, 0.75, 1.0),
    size=PHYSICS_SIZE // 2, seed=0,
))
rec.to_csv(f"{OUT}/physics_channel_recovery.csv", index=False)

summary = rec.groupby("severity").agg(
    psf_true=("psf_sigma_true", "median"),
    psf_est=("psf_sigma_est", "median"),
    psf_rel_err=("psf_sigma_rel_err", "median"),
    veil_true=("veil_fraction_true", "median"),
    veil_est=("veil_fraction_est", "median"),
    density_diff_rmse=("density_diff_rmse", "median"),
    density_abs_rmse=("density_abs_rmse", "median"),
).round(3)
display(summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

axes[0].plot(rec["psf_sigma_true"], rec["psf_sigma_est"], "o", alpha=0.6)
lim = [0, max(rec["psf_sigma_true"].max(), rec["psf_sigma_est"].max()) * 1.1]
axes[0].plot(lim, lim, "k--", lw=1)
axes[0].set_xlim(lim)
axes[0].set_ylim(lim)
axes[0].set_xlabel("true PSF sigma (px)")
axes[0].set_ylabel("recovered")
axes[0].set_title("PSF, from the slanted edge")

axes[1].plot(rec["veil_fraction_true"], rec["veil_fraction_est"], "o", alpha=0.6)
lim = [0, max(rec["veil_fraction_true"].max(), rec["veil_fraction_est"].max()) * 1.1]
axes[1].plot(lim, lim, "k--", lw=1)
axes[1].set_xlim(lim)
axes[1].set_ylim(lim)
axes[1].set_xlabel("true veil fraction")
axes[1].set_ylabel("recovered")
axes[1].set_title("Veiling glare, from the beam stop")

for col, lbl in (("density_diff_rmse", "differential (what the floor bounds)"),
                 ("density_abs_rmse", "absolute (gamma-prior limited)")):
    axes[2].plot(summary.index, summary[col], "o-", label=lbl)
axes[2].set_xlabel("capture severity")
axes[2].set_ylabel("density RMSE")
axes[2].set_title("Density recovery")
axes[2].legend(fontsize=7)

fig.tight_layout()
plt.show()

## 4. Detection, scored separately from estimation

A failure to *find* a fiducial and a failure to *estimate* from it need different fixes, and
lumping them together hides which is happening. This scores the detector itself: marker IoU
against the true marker mask, and collimation-corner error in pixels.

In [ ]:
fid = pd.DataFrame(V.fiducial_recovery(
    n_images=max(2, N_IMAGES // 6), size=PHYSICS_SIZE // 2, seed=1,
    severities=(0.0, 0.25, 0.5, 0.75, 1.0),
))
fid.to_csv(f"{OUT}/physics_fiducial_recovery.csv", index=False)

display(fid.groupby("severity").agg(
    corner_err_px=("corner_err_px", "median"),
    marker_iou=("marker_iou", "median"),
    marker_conf=("marker_confidence", "median"),
    mtf_edges=("n_mtf_edges", "median"),
    full_coverage=("coverage", lambda s: (s == "full").mean()),
).round(3))

### Reading this honestly

The collimation quad is recovered to well under a pixel on a clean capture and degrades
gracefully. The **marker is the fragile one**: blur merges the glyph into its surround and
glare washes it out, so it is often lost above moderate severity, and the detector is tuned to
reject rather than to guess (see `MARKER_ACCEPT` in `physics/fiducials.py` — a false marker
feeds the illumination fit a bogus interior sample, which is worse than having none).

Losing it costs less than it sounds. The unexposed film outside the collimation border is the
*same* `D_min` density and wraps the whole frame, so the density scale survives; the marker's unique contribution is being the only *interior* sample of the lightbox illumination.

**Known optimism, stated plainly.** The beam stop is an annulus, so the veil is measured around
the edge of the field and interpolated across the middle. A specular reflection sitting in the
centre of the film is caught only when it is bright enough to push a pixel above base+fog
(`glare._add_impossible_brightness` — nothing on a developed sheet is clearer than base+fog, so
any excess is unambiguously stray light). A dimmer central reflection is under-reported, and
the certificate is correspondingly optimistic there. This is the leading known bias in the
bound and is listed in `docs/PHYSICS.md`.